# Healthcare Agent Learning Lab 7/10

## Compounding reliability

**Suggested time:** 20 minutes  
**Demonstration type:** Deterministic mathematical simulation; no model

### Learning objectives

- Explain why end-to-end reliability differs from per-step accuracy.
- Identify high-consequence and correlated failure points.
- Relate verification placement to overall workflow performance.

### How to use this notebook

1. **Observe:** run the supplied demonstration and read the visible trajectory.
2. **Exercise:** discuss workflow, evidence, responsibility, and safety—no programming required.
3. **Try it:** edit only the clearly marked values in the exercise cell.
4. **Technical extension:** optional implementation work for technical participants.
5. **Reflect:** record what should be changed before the pattern is used in a real workflow.

> **Mode and safety notice:** Set `LAB_MODE` explicitly to `mock` or `live`; the notebook will stop rather than silently choose a mode. Mock mode is scripted and demonstrates system structure—it is not evidence of model performance. Live mode may vary between runs and may incur API cost. All patients, actions, protocols, and outputs in this curriculum are fictional or synthetic and must not be used for patient care.


## Demonstration
**Goal:** *measure* the effect behind Figure 6 of the white paper: steps that are each ~95% reliable compound to a much lower end-to-end reliability — and a benchmark that scores one step in isolation will overstate readiness.

This notebook is **pure simulation** — no model or API key needed. It's about measurement, not generation.


> ### In plain terms
> Here's a counter-intuitive fact worth feeling in your gut: if a task has five steps and each step is **95% reliable**, the *whole task* is only about **77%** reliable (0.95 × 0.95 × … five times). A test that checks one step in isolation looks far rosier than what the patient actually experiences across the whole chain.
>
> **Note:** no AI or API key here — it's a small simulation about *measurement*.
>
> **What you'll see:** reliability "leaking" step by step (the bar chart), and then how a verifier/gate that catches some errors buys reliability back.


In [ ]:
import random, statistics
random.seed(0)

STEPS = ["Intake", "Retrieve", "Interpret", "Recommend", "Act / document"]
P = 0.95                     # per-step reliability (each step is quite good on its own)
N = 20000                    # simulated patient episodes

def run_episode(p=P):
    "An episode succeeds only if EVERY step succeeds."
    return all(random.random() < p for _ in STEPS)

end_to_end = statistics.mean(run_episode() for _ in range(N))
print(f"Per-step reliability     : {P:.2f}  (what a single-step benchmark sees)")
print(f"Steps in the workflow    : {len(STEPS)}")
print(f"End-to-end (simulated)   : {end_to_end:.2f}  over {N:,} episodes")
print(f"End-to-end (analytic)    : {P**len(STEPS):.2f}  = {P}^{len(STEPS)}")


### Where the reliability leaks: the cumulative curve
Each step multiplies what survives. This is the curve in Figure 6.


In [ ]:
cum = 1.0
print(f"{'after step':<18}{'cumulative reliability':>22}")
for s in STEPS:
    cum *= P
    bar = "#" * round(cum * 40)
    print(f"{s:<18}{cum:>10.2f}   {bar}")
print(f"\nA benchmark scoring only 'Interpret' reports ~{P:.2f}.")
print(f"The patient experiences the whole chain: ~{cum:.2f}.")


### Two things your team should feel
1. **Longer workflows leak more** — reliability falls as the chain grows.
2. **A verifier/gate buys back reliability** — catching and fixing a fraction of step errors raises the effective per-step rate, which compounds in your favour.


In [ ]:
# 1) end-to-end vs. chain length, at a few per-step rates
print("chain length ->     3       5       8")
for p in [0.99, 0.95, 0.90]:
    row = "   ".join(f"{p**k:5.2f}" for k in (3, 5, 8))
    print(f"  per-step {p:.2f}:   {row}")

# 2) a verifier that catches a fraction c of the errors at each step
def effective_p(p, c):        # c = share of step errors caught & fixed
    return p + (1 - p) * c
print("\n5-step end-to-end, per-step 0.95:")
for c in [0.0, 0.5, 0.8, 0.95]:
    ep = effective_p(0.95, c)
    print(f"  verifier catches {c:>4.0%} of errors -> effective step {ep:.3f} -> end-to-end {ep**5:.2f}")


### Takeaway
Nothing here is pessimism about any single component — every step was 0.95. The point is structural: **the metric has to match what the patient experiences.** Evaluate the consequential end-state across the whole trajectory, not one step in isolation, and treat verification/gating as a reliability lever, not just a safety one. This is why the paper argues for process- and outcome-level, ambiguity-stratified evaluation rather than a single benchmark score.

*Try:* change `P`, add or remove a step in `STEPS`, or raise the verifier's catch-rate, and watch the end-to-end number move.


## Exercise

No programming is required for this section.

**Activity:** Map a familiar clinical workflow into steps. Assign each step a plausible failure risk and consequence, then decide where verification or human review provides the greatest value.

### Discussion prompts

- Which failure ends the workflow even if every later step is correct?
- Are errors likely to be independent in a real EHR workflow?
- Where would a verifier reduce risk most efficiently?

Record the group’s proposed workflow boundary, the evidence needed, and the person or role accountable for the consequential decision.


In [ ]:
# CLINICAL TRY-IT: change the step reliabilities to match a workflow you know.
STEP_RELIABILITIES = [0.99, 0.95, 0.97, 0.90, 0.98]
end_to_end = 1.0
for index, probability in enumerate(STEP_RELIABILITIES, start=1):
    end_to_end *= probability
    print(f"After step {index}: {end_to_end:.1%}")
print("Assumption: step errors are independent. Real workflows may violate this.")


## Optional technical extension

Extend the independent-error simulation with unequal step probabilities and a shared failure factor. Compare analytical results with Monte Carlo confidence intervals.

Technical participants should report both successful behavior and at least one intentionally induced failure. Preserve the trace needed to explain why the failure occurred.


## Interpretation boundary

The multiplication example assumes independent step success. Use it as an intuition builder, then explicitly discuss correlated errors, unequal consequences, recovery, and conditional branching.

### Exit question

In one sentence: **What should the environment enforce, rather than merely asking the model to do?**

Continue to the next notebook only after the group can distinguish the observed system behavior from any claim about clinical correctness.
